
# Slither.io version of `humand_data`

This notebook is a Slither.io-oriented copy/adaptation of the repository's `humand_data.ipynb` workflow:

1. load game data,
2. build a connectome-style reservoir,
3. collect reservoir states,
4. fit a ridge-regression readout,
5. evaluate angle/boost prediction.

It is written to prefer real scraper data from a local `data/` folder next to the notebook. If no such data is present, it creates a tiny schema-compatible mock dataset so the pipeline can be smoke-tested end to end.


In [ ]:

from pathlib import Path
from typing import List, Dict, Optional, Tuple
import json
import math
import os
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

np.random.seed(7)
rng = np.random.default_rng(7)

import sys
from pathlib import Path


In [ ]:
# --- configuration ---
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
GENERATED_DIR = BASE_DIR / "generated_artifacts"
GENERATED_DIR.mkdir(exist_ok=True)

ANGLE_MIN = -40
ANGLE_MAX = 40
ANGLE_RESOLUTION = 5
NUM_ANGLE_BINS = int((ANGLE_MAX - ANGLE_MIN) / ANGLE_RESOLUTION) + 1
WINDOW_LEN = 25
WINDOW_STRIDE = 15
MIN_FRAMES_PER_SESSION = 100
TEST_RATIO = 0.25
ALPHA = 1e-2
T_WASHOUT = 5
LEAK_RANGE = (0.1, 0.3)
INPUT_SCALE = 0.6

In [ ]:
def _maybe_import_zarr():
    try:
        import zarr
        return zarr
    except Exception:
        return None


def discover_sessions(data_path: Path) -> List[Path]:
    sessions = []
    if not data_path.exists():
        return sessions
    for user_dir in sorted(data_path.iterdir()):
        if not user_dir.is_dir():
            continue
        for item in sorted(user_dir.iterdir()):
            if not item.is_dir():
                continue
            if item.name.startswith("session_") or item.name.startswith("game_"):
                if (item / "grids.npy").exists() or (item / "grids").exists():
                    sessions.append(item)
    return sessions


def load_session(session_path: Path) -> Optional[Dict[str, np.ndarray]]:
    arrays = [
        "grids", "player_inputs", "timestamps", "headings",
        "velocities", "distances_to_border", "boost_states"
    ]

    # lightweight .npy mock/demo format
    if (session_path / "grids.npy").exists():
        data = {name: np.load(session_path / f"{name}.npy") for name in arrays}
    else:
        zarr = _maybe_import_zarr()
        if zarr is None:
            raise RuntimeError(
                "This notebook can read real scraper sessions through zarr, but zarr is not installed in this environment. "
                "Either install zarr or let the notebook use the built-in mock .npy demo data."
            )
        try:
            try:
                root = zarr.open_group(str(session_path), mode='r')
            except Exception:
                store = zarr.DirectoryStore(str(session_path))
                root = zarr.group(store=store)
            data = {name: np.array(root[name]) for name in arrays}
        except Exception as e:
            print(f"Skipping {session_path.name}: {e}")
            return None

    metadata = {}
    meta_file = session_path / "metadata.json"
    if meta_file.exists():
        metadata = json.loads(meta_file.read_text())

    n_frames = int(data["grids"].shape[0])
    if n_frames < MIN_FRAMES_PER_SESSION:
        print(f"Skipping {session_path.name}: only {n_frames} frames")
        return None

    data["metadata"] = metadata
    data["session_path"] = session_path
    return data


def normalize_grids(grids: np.ndarray) -> np.ndarray:
    return np.clip(grids.astype(np.float32), 0.0, 1.0)


def prepare_features(session_data: Dict[str, np.ndarray]) -> np.ndarray:
    n_frames = session_data["grids"].shape[0]
    grids_flat = normalize_grids(session_data["grids"]).reshape(n_frames, -1)

    velocity = session_data["velocities"].reshape(-1, 1).astype(np.float32) / 200.0
    distance = session_data["distances_to_border"].reshape(-1, 1).astype(np.float32) / 21600.0

    headings = session_data["headings"].astype(np.float32)
    angular_vel = np.diff(headings, prepend=headings[0])
    angular_vel = np.arctan2(np.sin(angular_vel), np.cos(angular_vel)) / np.pi
    angular_vel = angular_vel.reshape(-1, 1)

    mx = session_data["player_inputs"][:, 0].astype(np.float32)
    my = session_data["player_inputs"][:, 1].astype(np.float32)
    prev_angle = np.arctan2(my, mx)
    prev_angle = np.roll(prev_angle, 1)
    prev_angle[0] = 0.0
    prev_sin = np.sin(prev_angle).reshape(-1, 1)
    prev_cos = np.cos(prev_angle).reshape(-1, 1)

    X = np.concatenate([
        grids_flat,
        velocity,
        distance,
        angular_vel,
        prev_sin,
        prev_cos,
    ], axis=1)
    return X.astype(np.float32)


def convert_to_angle_bins(player_inputs: np.ndarray) -> np.ndarray:
    mx = player_inputs[:, 0]
    my = player_inputs[:, 1]
    boost = player_inputs[:, 2]

    angles_deg = np.degrees(np.arctan2(my, mx))
    angles_deg = np.clip(angles_deg, ANGLE_MIN, ANGLE_MAX)
    positions = (angles_deg - ANGLE_MIN) / ANGLE_RESOLUTION
    positions = np.clip(positions, 0, NUM_ANGLE_BINS - 1)

    y_angle = np.zeros((len(player_inputs), NUM_ANGLE_BINS), dtype=np.float32)
    for i, pos in enumerate(positions):
        lo = int(np.floor(pos))
        hi = int(np.ceil(pos))
        lo = np.clip(lo, 0, NUM_ANGLE_BINS - 1)
        hi = np.clip(hi, 0, NUM_ANGLE_BINS - 1)
        if lo == hi:
            y_angle[i, lo] = 1.0
        else:
            w_hi = pos - lo
            w_lo = 1.0 - w_hi
            y_angle[i, lo] = w_lo
            y_angle[i, hi] = w_hi

    return np.concatenate([y_angle, boost.reshape(-1, 1).astype(np.float32)], axis=1)


def load_all_sessions(data_dir: Path):
    X_list, y_list, names = [], [], []
    for session_path in discover_sessions(data_dir):
        session = load_session(session_path)
        if session is None:
            continue
        X_list.append(prepare_features(session))
        y_list.append(convert_to_angle_bins(session["player_inputs"]))
        names.append(session_path.name)
    return X_list, y_list, names


def make_windows(X_list, y_list, window_len=WINDOW_LEN, stride=WINDOW_STRIDE):
    u, y = [], []
    session_ids = []
    for sid, (X, Y) in enumerate(zip(X_list, y_list)):
        if len(X) < window_len:
            continue
        for start in range(0, len(X) - window_len + 1, stride):
            stop = start + window_len
            u.append(X[start:stop])
            y.append(Y[start:stop])
            session_ids.append(sid)
    return np.array(u, dtype=np.float32), np.array(y, dtype=np.float32), np.array(session_ids)


def train_test_split_windows(u, y, session_ids, test_ratio=TEST_RATIO, seed=7):
    rng_local = np.random.default_rng(seed)
    idx = np.arange(len(u))
    rng_local.shuffle(idx)
    n_test = max(1, int(len(idx) * test_ratio))
    test_idx = np.sort(idx[:n_test])
    train_idx = np.sort(idx[n_test:])
    return u[train_idx], y[train_idx], u[test_idx], y[test_idx], session_ids[train_idx], session_ids[test_idx]


def compute_wout(X_train_states: np.ndarray, y_train: np.ndarray, washout: int = T_WASHOUT, alpha: float = ALPHA):
    X = X_train_states[:, washout:, :].reshape(-1, X_train_states.shape[-1])
    Y = y_train[:, washout:, :].reshape(-1, y_train.shape[-1])
    R = X.T @ X
    P = X.T @ Y
    return np.linalg.solve(R + alpha * np.eye(X.shape[1]), P)


def angle_accuracy(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    yp = y_pred[..., :NUM_ANGLE_BINS].reshape(-1, NUM_ANGLE_BINS).argmax(axis=1)
    yt = y_true[..., :NUM_ANGLE_BINS].reshape(-1, NUM_ANGLE_BINS).argmax(axis=1)
    return float(np.mean(yp == yt))


def boost_accuracy(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    yp = (y_pred[..., -1].reshape(-1) >= 0.5).astype(int)
    yt = (y_true[..., -1].reshape(-1) >= 0.5).astype(int)
    return float(np.mean(yp == yt))


In [ ]:
def ensure_mock_graph(graph_dir: Path):
    files = list(graph_dir.glob("*.graphml"))
    if files:
        return files
    G = nx.erdos_renyi_graph(60, 0.08, seed=7)
    for u, v in G.edges():
        G[u][v]["weight"] = float(rng.uniform(0.2, 1.0))
    out = graph_dir / "mock_connectome.graphml"
    nx.write_graphml(G, out)
    return [out]


def ensure_mock_data(data_dir: Path, n_sessions: int = 3, n_frames: int = 240):
    sessions = discover_sessions(data_dir)
    if sessions:
        return False

    for user_id in range(1, n_sessions + 1):
        session_path = data_dir / f"mock_user_{user_id}" / f"session_{1000 + user_id}"
        session_path.mkdir(parents=True, exist_ok=True)

        headings = np.cumsum(rng.normal(0.0, 0.12, size=n_frames)).astype(np.float32)
        velocity = np.clip(90 + 30 * np.sin(np.linspace(0, 4 * np.pi, n_frames)) + rng.normal(0, 7, n_frames), 20, 180).astype(np.float32)
        distance = np.clip(12000 + 4000 * np.sin(np.linspace(0, 2 * np.pi, n_frames) + user_id) + rng.normal(0, 300, n_frames), 1500, 21500).astype(np.float32)
        boost = (velocity > 110).astype(np.float32)

        mx = np.cos(headings)
        my = np.sin(headings)
        player_inputs = np.stack([mx, my, boost], axis=1).astype(np.float32)

        grids = np.zeros((n_frames, 64, 24, 4), dtype=np.float32)
        for t in range(n_frames):
            ang_idx = int(((np.degrees(headings[t]) + 180) / 360.0) * 64) % 64
            rad_idx = min(23, max(0, int((velocity[t] / 180.0) * 23)))
            grids[t, ang_idx, max(0, rad_idx - 1):min(24, rad_idx + 2), 0] = 1.0
            grids[t, (ang_idx + 3) % 64, :, 1] = np.linspace(0.2, 0.8, 24)
            grids[t, ang_idx, : max(2, rad_idx), 2] = 0.5
            grids[t, (ang_idx + 8) % 64, min(23, rad_idx + 2), 3] = 1.0
            grids[t] += rng.normal(0.0, 0.03, size=grids[t].shape)
        grids = np.clip(grids, 0.0, 1.0)

        timestamps = np.arange(n_frames, dtype=np.float32) * 0.1

        np.save(session_path / "grids.npy", grids)
        np.save(session_path / "player_inputs.npy", player_inputs)
        np.save(session_path / "timestamps.npy", timestamps)
        np.save(session_path / "headings.npy", headings)
        np.save(session_path / "velocities.npy", velocity)
        np.save(session_path / "distances_to_border.npy", distance)
        np.save(session_path / "boost_states.npy", boost)
        (session_path / "metadata.json").write_text(json.dumps({"username": f"mock_user_{user_id}", "source": "generated_demo"}))

    return True


In [ ]:

from reservoir import Reservoir, Reservoir2, Reservoir3

names = ("Reservoir", "Reservoir1", "Reservoir2")
objects = {
    "Reservoir": Reservoir,
    "Reservoir1": Reservoir2,
    "Reservoir2": Reservoir3,
}
r_how_ls=(0.5,0.75,0.9,0.95,1.05,1.1,1.25,1.5)
neurons=(1000,100,250,500,24)

for i in range(3):
    current = objects[names[i]]
    opt_neurons=0
    opt_true_bin=0
    opt_pred_bin=0
    opt_mse=1
    opt_rho=0
    for N_NEURONS in neurons:
        for RHOW in r_how_ls:
                session_paths = discover_sessions(DATA_DIR)
                sources = []
                for sp in session_paths:
                    meta_file = sp / "metadata.json"
                    meta = json.loads(meta_file.read_text()) if meta_file.exists() else {}
                    sources.append(meta.get("source", "unknown"))
                using_generated_only = len(session_paths) > 0 and all(src == "generated_demo" for src in sources)
                real_sessions_present = len(session_paths) > 0 and not using_generated_only
                
                X_list, y_list, session_names = load_all_sessions(DATA_DIR)

                u, y, session_ids = make_windows(X_list, y_list, window_len=WINDOW_LEN, stride=WINDOW_STRIDE)
                u_train, y_train, u_test, y_test, sid_train, sid_test = train_test_split_windows(u, y, session_ids, test_ratio=TEST_RATIO, seed=7)

                reservoir = Reservoir(n_inputs=u_train.shape[2], n_neurons=N_NEURONS, rhow=RHOW, leak_range=LEAK_RANGE)

                
                def collect_states_batch(reservoir: Reservoir, u_batch: np.ndarray) -> np.ndarray:
                    return np.stack([reservoir.forward(seq, collect_states=True) for seq in u_batch], axis=0)

                X_train_states = collect_states_batch(reservoir, u_train)
                X_test_states = collect_states_batch(reservoir, u_test)

                
                wout = compute_wout(X_train_states, y_train, washout=T_WASHOUT, alpha=ALPHA)
                    
                    
                y_pred_train = X_train_states @ wout
                y_pred_test = X_test_states @ wout

                metrics = pd.DataFrame([
                    {
                        "split": "train",
                        "angle_accuracy": angle_accuracy(y_pred_train, y_train),
                        "boost_accuracy": boost_accuracy(y_pred_train, y_train),
                        "mse": float(np.mean((y_pred_train - y_train) ** 2)),
                    },
                    {
                        "split": "test",
                        "angle_accuracy": angle_accuracy(y_pred_test, y_test),
                        "boost_accuracy": boost_accuracy(y_pred_test, y_test),
                        "mse": float(np.mean((y_pred_test - y_test) ** 2)),
                    },
                ])
                metrics

                # quick visual check on one held-out window
                angles = np.arange(ANGLE_MIN, ANGLE_MAX + ANGLE_RESOLUTION, ANGLE_RESOLUTION)
                idx = 0
                true_bins = y_test[idx, :, :NUM_ANGLE_BINS].argmax(axis=1)
                pred_bins = y_pred_test[idx, :, :NUM_ANGLE_BINS].argmax(axis=1)
                if metrics[metrics['split'] == 'test']['mse'].iloc[0] <= opt_mse:
                    opt_mse=metrics[metrics['split'] == 'test']['mse'].iloc[0]
                    opt_neurons=N_NEURONS
                    opt_rho=RHOW
                    opt_true_bin=true_bins
                    opt_pred_bin=pred_bins
                    
    print(f"Best config: neurons={opt_neurons}, rho={opt_rho}, mse={opt_mse:.4f}")
    print(metrics.to_string(index=False))

    plt.figure(figsize=(10, 4))
    plt.plot(angles[true_bins], label="true angle bin")
    plt.plot(angles[pred_bins], label="pred angle bin")
    plt.xlabel("time step in held-out window")
    plt.ylabel("degrees")
    plt.title("Held-out Slither.io window: true vs predicted angle bins")
    plt.legend()
    plt.tight_layout()
    plt.show()

    print("True boost (first 10):", y_test[idx, :10, -1].round(3))
    print("Pred boost (first 10):", y_pred_test[idx, :10, -1].round(3))
    print("----------------------------------------------------------------------------------------------------------------")




Graph dir used: C:\Users\ricca\OneDrive\universita\BAINSA\ESN\dati\network_architecture\folder_path_83


In [ ]:
""" created_mock_data = ensure_mock_data(DATA_DIR)
mock_graph_files = ensure_mock_graph(GRAPH_DIR)
session_paths = discover_sessions(DATA_DIR)
sources = []
for sp in session_paths:
    meta_file = sp / "metadata.json"
    meta = json.loads(meta_file.read_text()) if meta_file.exists() else {}
    sources.append(meta.get("source", "unknown"))
using_generated_only = len(session_paths) > 0 and all(src == "generated_demo" for src in sources)
real_sessions_present = len(session_paths) > 0 and not using_generated_only

print(f"Base dir: {BASE_DIR}")
print(f"Data dir: {DATA_DIR}")
print(f"Graph dir used: {GRAPH_DIR}")
print(f"Sessions discovered: {len(session_paths)}")
print(f"Using real scraper data: {real_sessions_present}")
print(f"Using generated demo data only: {using_generated_only}")
print(f"Created mock demo data in this run: {created_mock_data}")
print(f"Graph files: {[p.name for p in mock_graph_files]}") """

' created_mock_data = ensure_mock_data(DATA_DIR)\nmock_graph_files = ensure_mock_graph(GRAPH_DIR)\nsession_paths = discover_sessions(DATA_DIR)\nsources = []\nfor sp in session_paths:\n    meta_file = sp / "metadata.json"\n    meta = json.loads(meta_file.read_text()) if meta_file.exists() else {}\n    sources.append(meta.get("source", "unknown"))\nusing_generated_only = len(session_paths) > 0 and all(src == "generated_demo" for src in sources)\nreal_sessions_present = len(session_paths) > 0 and not using_generated_only\n\nprint(f"Base dir: {BASE_DIR}")\nprint(f"Data dir: {DATA_DIR}")\nprint(f"Graph dir used: {GRAPH_DIR}")\nprint(f"Sessions discovered: {len(session_paths)}")\nprint(f"Using real scraper data: {real_sessions_present}")\nprint(f"Using generated demo data only: {using_generated_only}")\nprint(f"Created mock demo data in this run: {created_mock_data}")\nprint(f"Graph files: {[p.name for p in mock_graph_files]}") '


## Load the Slither.io sessions

This mirrors the first part of the original notebook, but instead of transforming Iris features into sinusoidal signals, it loads frame-wise Slither.io state tensors and labels.


In [ ]:
""" 
X_list, y_list, session_names = load_all_sessions(DATA_DIR)

print("Loaded sessions:", session_names)
for name, X, y in zip(session_names, X_list, y_list):
    print(f"- {name}: X={X.shape}, y={y.shape}")

u, y, session_ids = make_windows(X_list, y_list, window_len=WINDOW_LEN, stride=WINDOW_STRIDE)
print("\nWindowed dataset")
print("u shape:", u.shape)
print("y shape:", y.shape)
print("session_ids shape:", session_ids.shape)
 """

' \nX_list, y_list, session_names = load_all_sessions(DATA_DIR)\n\nprint("Loaded sessions:", session_names)\nfor name, X, y in zip(session_names, X_list, y_list):\n    print(f"- {name}: X={X.shape}, y={y.shape}")\n\nu, y, session_ids = make_windows(X_list, y_list, window_len=WINDOW_LEN, stride=WINDOW_STRIDE)\nprint("\nWindowed dataset")\nprint("u shape:", u.shape)\nprint("y shape:", y.shape)\nprint("session_ids shape:", session_ids.shape)\n '

In [ ]:
""" 
u_train, y_train, u_test, y_test, sid_train, sid_test = train_test_split_windows(u, y, session_ids, test_ratio=TEST_RATIO, seed=7)

print("Training windows:", u_train.shape)
print("Testing windows:", u_test.shape)
print("Input dim:", u_train.shape[-1])
print("Output dim:", y_train.shape[-1])
 """

' \nu_train, y_train, u_test, y_test, sid_train, sid_test = train_test_split_windows(u, y, session_ids, test_ratio=TEST_RATIO, seed=7)\n\nprint("Training windows:", u_train.shape)\nprint("Testing windows:", u_test.shape)\nprint("Input dim:", u_train.shape[-1])\nprint("Output dim:", y_train.shape[-1])\n '


## Set up the reservoir

As in `humand_data.ipynb`, we build a reservoir first and only train the readout layer.


In [ ]:
""" 
reservoir = ConnectomeReservoir(
    n_inputs=u_train.shape[2],
    graph_dir=str(GRAPH_DIR),
    edge_attr="weight",
    combine="mean",
    rhow=RHOW,
    leak_range=LEAK_RANGE,
    symmetric=True,
    seed=7,
    input_scale=INPUT_SCALE,
    target_size=N_NEURONS,
)

print(f"Reservoir neurons: {reservoir.n_neurons}")
 """

' \nreservoir = ConnectomeReservoir(\n    n_inputs=u_train.shape[2],\n    graph_dir=str(GRAPH_DIR),\n    edge_attr="weight",\n    combine="mean",\n    rhow=RHOW,\n    leak_range=LEAK_RANGE,\n    symmetric=True,\n    seed=7,\n    input_scale=INPUT_SCALE,\n    target_size=N_NEURONS,\n)\n\nprint(f"Reservoir neurons: {reservoir.n_neurons}")\n '

In [ ]:
""" 
def collect_states_batch(reservoir: ConnectomeReservoir, u_batch: np.ndarray) -> np.ndarray:
    return np.stack([reservoir.forward(seq, collect_states=True) for seq in u_batch], axis=0)

X_train_states = collect_states_batch(reservoir, u_train)
X_test_states = collect_states_batch(reservoir, u_test)

print("X_train_states:", X_train_states.shape)
print("X_test_states:", X_test_states.shape)
 """

' \ndef collect_states_batch(reservoir: ConnectomeReservoir, u_batch: np.ndarray) -> np.ndarray:\n    return np.stack([reservoir.forward(seq, collect_states=True) for seq in u_batch], axis=0)\n\nX_train_states = collect_states_batch(reservoir, u_train)\nX_test_states = collect_states_batch(reservoir, u_test)\n\nprint("X_train_states:", X_train_states.shape)\nprint("X_test_states:", X_test_states.shape)\n '


## Train the readout map

We fit a ridge-regression readout on the reservoir states after a short washout period.


In [ ]:
""" 
wout = compute_wout(X_train_states, y_train, washout=T_WASHOUT, alpha=ALPHA)
print("wout shape:", wout.shape)
 """

' \nwout = compute_wout(X_train_states, y_train, washout=T_WASHOUT, alpha=ALPHA)\nprint("wout shape:", wout.shape)\n '

In [ ]:
""" 
y_pred_train = X_train_states @ wout
y_pred_test = X_test_states @ wout

metrics = pd.DataFrame([
    {
        "split": "train",
        "angle_accuracy": angle_accuracy(y_pred_train, y_train),
        "boost_accuracy": boost_accuracy(y_pred_train, y_train),
        "mse": float(np.mean((y_pred_train - y_train) ** 2)),
    },
    {
        "split": "test",
        "angle_accuracy": angle_accuracy(y_pred_test, y_test),
        "boost_accuracy": boost_accuracy(y_pred_test, y_test),
        "mse": float(np.mean((y_pred_test - y_test) ** 2)),
    },
])
metrics

print(metrics.to_string(index=False))
 """

' \ny_pred_train = X_train_states @ wout\ny_pred_test = X_test_states @ wout\n\nmetrics = pd.DataFrame([\n    {\n        "split": "train",\n        "angle_accuracy": angle_accuracy(y_pred_train, y_train),\n        "boost_accuracy": boost_accuracy(y_pred_train, y_train),\n        "mse": float(np.mean((y_pred_train - y_train) ** 2)),\n    },\n    {\n        "split": "test",\n        "angle_accuracy": angle_accuracy(y_pred_test, y_test),\n        "boost_accuracy": boost_accuracy(y_pred_test, y_test),\n        "mse": float(np.mean((y_pred_test - y_test) ** 2)),\n    },\n])\nmetrics\n\nprint(metrics.to_string(index=False))\n '

In [ ]:
""" 
# quick visual check on one held-out window
angles = np.arange(ANGLE_MIN, ANGLE_MAX + ANGLE_RESOLUTION, ANGLE_RESOLUTION)
idx = 0
true_bins = y_test[idx, :, :NUM_ANGLE_BINS].argmax(axis=1)
pred_bins = y_pred_test[idx, :, :NUM_ANGLE_BINS].argmax(axis=1)

plt.figure(figsize=(10, 4))
plt.plot(angles[true_bins], label="true angle bin")
plt.plot(angles[pred_bins], label="pred angle bin")
plt.xlabel("time step in held-out window")
plt.ylabel("degrees")
plt.title("Held-out Slither.io window: true vs predicted angle bins")
plt.legend()
plt.tight_layout()
plt.show()

print("True boost (first 10):", y_test[idx, :10, -1].round(3))
print("Pred boost (first 10):", y_pred_test[idx, :10, -1].round(3))
 """

' \n# quick visual check on one held-out window\nangles = np.arange(ANGLE_MIN, ANGLE_MAX + ANGLE_RESOLUTION, ANGLE_RESOLUTION)\nidx = 0\ntrue_bins = y_test[idx, :, :NUM_ANGLE_BINS].argmax(axis=1)\npred_bins = y_pred_test[idx, :, :NUM_ANGLE_BINS].argmax(axis=1)\n\nplt.figure(figsize=(10, 4))\nplt.plot(angles[true_bins], label="true angle bin")\nplt.plot(angles[pred_bins], label="pred angle bin")\nplt.xlabel("time step in held-out window")\nplt.ylabel("degrees")\nplt.title("Held-out Slither.io window: true vs predicted angle bins")\nplt.legend()\nplt.tight_layout()\nplt.show()\n\nprint("True boost (first 10):", y_test[idx, :10, -1].round(3))\nprint("Pred boost (first 10):", y_pred_test[idx, :10, -1].round(3))\n '


## Notes

- If you place real scraper data under `data/<user>/session_*` or `data/<user>/game_*`, the notebook will use that instead of the generated demo data.
- If you place real `.graphml` connectome files in `generated_artifacts/graphs` or adjust `GRAPH_DIR`, the notebook will use those.
- The current environment used for this smoke test did not have the branch's real `data/` folder available, so the executed outputs below come from the schema-compatible generated demo dataset.


234 nodes:

Windowed dataset
u shape: (45, 25, 6149)
y shape: (45, 25, 18)
session_ids shape: (45,)
Training windows: (34, 25, 6149)
Testing windows: (11, 25, 6149)
Input dim: 6149
Output dim: 18
Reservoir neurons: 1000
X_train_states: (34, 25, 1000)
X_test_states: (11, 25, 1000)
wout shape: (1000, 18)
split  angle_accuracy  boost_accuracy      mse
train        0.965882        0.982353 0.005809
 test        0.825455        0.901818 0.053515

1015 nodes:

split  angle_accuracy  boost_accuracy      mse
train        0.965882        0.974118 0.006131
 test        0.730909        0.814545 0.064433